In [1]:
import os

In [2]:
import torch

In [3]:
import torch.nn as nn

In [4]:
import torch.optim as optim

In [5]:
from torchvision import datasets, transforms, models

In [6]:
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

C:\ProgramData\Anaconda3-2022\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [8]:
import random

In [9]:
from collections import Counter

In [10]:
DATASET_PATH = "../Dataset/"
MODEL_PATH = "vit_fast.pth"
DEVICE = torch.device("cpu") 

In [11]:
BATCH_SIZE = 8
EPOCHS = 3
IMG_SIZE = 64  

In [12]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [13]:
dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = dataset.classes

In [14]:
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size

In [15]:
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [16]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [17]:
class FastViT(nn.Module):
    def __init__(self, img_size=64, patch_size=8, num_classes=4, dim=64, depth=2, heads=4):
        super().__init__()

        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = 3 * patch_size * patch_size

        self.patch_embed = nn.Linear(self.patch_dim, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.fc = nn.Linear(dim, num_classes)

    def forward(self, x):
        B, C, H, W = x.shape

        patches = x.unfold(2, self.patch_size, self.patch_size)\
                   .unfold(3, self.patch_size, self.patch_size)

        patches = patches.contiguous().view(B, C, -1, self.patch_size, self.patch_size)
        patches = patches.permute(0, 2, 1, 3, 4)
        patches = patches.contiguous().view(B, self.num_patches, -1)

        x = self.patch_embed(patches)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed

        x = self.transformer(x)
        x = x[:, 0]

        return self.fc(x)

In [18]:
model = FastViT(num_classes=len(class_names)).to(DEVICE)

C:\ProgramData\Anaconda3-2022\lib\site-packages\torch\nn\modules\transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
can_train = False

In [21]:
if can_train:
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")
        torch.save(model.state_dict(), MODEL_PATH)

In [22]:
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

FastViT(
  (patch_embed): Linear(in_features=192, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=4, bias=True)
)

In [23]:
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy())

In [24]:
print("\n===== METRICS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='weighted'))
print("Recall   :", recall_score(y_true, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_true, y_pred, average='weighted'))


===== METRICS =====
Accuracy : 0.7749113064939072
Precision: 0.6004875329320941
Recall   : 0.7749113064939072
F1 Score : 0.67663948134769


C:\ProgramData\Anaconda3-2022\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
print("\n===== SAMPLE PREDICTIONS =====")

for i in random.sample(range(len(test_dataset)), 3):
    img, label = test_dataset[i]
    with torch.no_grad():
        pred = model(img.unsqueeze(0))
        _, pred_class = torch.max(pred, 1)

    print("\nActual   :", class_names[label])
    print("Predicted:", class_names[pred_class.item()])


===== SAMPLE PREDICTIONS =====

Actual   : Non Demented
Predicted: Non Demented

Actual   : Non Demented
Predicted: Non Demented

Actual   : Very mild Dementia
Predicted: Non Demented


In [26]:
DATASET_PATH = "../Dataset/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:
BATCH_SIZE = 8
EPOCHS = 5
IMG_SIZE = 64

In [28]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [29]:
dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = dataset.classes
num_classes = len(class_names)


In [30]:
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size

In [31]:
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [32]:
train_labels = [dataset.targets[i] for i in train_dataset.indices]

In [33]:
class_counts = Counter(train_labels)
total_samples = len(train_labels)

In [34]:
class_weights = []
for i in range(num_classes):
    class_weights.append(total_samples / (num_classes * class_counts[i]))

In [35]:
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

In [36]:
sample_weights = [class_weights[label].item() for label in train_labels]

In [37]:
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

In [38]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [39]:
model = models.densenet121(pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, num_classes)
model = model.to(DEVICE)

C:\ProgramData\Anaconda3-2022\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\Anaconda3-2022\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [40]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [41]:
if can_train:
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(DEVICE))
            _, preds = torch.max(outputs, 1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    print("\n===== METRICS =====")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_true, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_true, y_pred, average='weighted'))
    
    print("\n===== SAMPLE PREDICTIONS =====")

    for i in random.sample(range(len(test_dataset)), 3):
        img, label = test_dataset[i]
        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(DEVICE))
            _, pred_class = torch.max(pred, 1)

        print("\nActual   :", class_names[label])
        print("Predicted:", class_names[pred_class.item()])

In [42]:
DATASET_PATH = "../Dataset/"
DEVICE = torch.device("cpu")

In [43]:
BATCH_SIZE = 16
EPOCHS = 5
IMG_SIZE = 64

In [44]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [45]:
dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = dataset.classes
num_classes = len(class_names)

In [46]:
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size

In [47]:
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [48]:
train_labels = [dataset.targets[i] for i in train_dataset.indices]

In [49]:
class_counts = Counter(train_labels)
total_samples = len(train_labels)

In [50]:
class_weights = [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(class_weights, dtype=torch.float)

In [51]:
sample_weights = [class_weights[label].item() for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [52]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [53]:
class FineGrainedNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = FineGrainedNet(num_classes)

In [54]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [55]:
MODEL_PATH = "fine_grained_net_model.pth"

In [56]:
if can_train:
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")
    torch.save({
        'model_state_dict': model.state_dict(),
        'num_classes': num_classes
    }, MODEL_PATH)

In [57]:
checkpoint = torch.load(MODEL_PATH, map_location=torch.device("cpu"))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

FineGrainedNet(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)

In [58]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy())

In [59]:
print("\n===== METRICS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='weighted'))
print("Recall   :", recall_score(y_true, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_true, y_pred, average='weighted'))


===== METRICS =====
Accuracy : 0.758792225821379
Precision: 0.8984358450134936
Recall   : 0.758792225821379
F1 Score : 0.7934201476336024


In [60]:
print("\n===== SAMPLE PREDICTIONS =====")

for i in random.sample(range(len(test_dataset)), 3):
    img, label = test_dataset[i]
    with torch.no_grad():
        pred = model(img.unsqueeze(0))
        _, pred_class = torch.max(pred, 1)

    print("\nActual   :", class_names[label])
    print("Predicted:", class_names[pred_class.item()])


===== SAMPLE PREDICTIONS =====

Actual   : Very mild Dementia
Predicted: Very mild Dementia

Actual   : Non Demented
Predicted: Non Demented

Actual   : Very mild Dementia
Predicted: Very mild Dementia


In [61]:
DATASET_PATH = "../Dataset/"
DEVICE = torch.device("cpu")

In [62]:
BATCH_SIZE = 16
EPOCHS = 5
IMG_SIZE = 64

In [63]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [64]:
dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = dataset.classes
num_classes = len(class_names)

In [65]:
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size

In [66]:
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [67]:
train_labels = [dataset.targets[i] for i in train_dataset.indices]

In [68]:
class_counts = Counter(train_labels)
total_samples = len(train_labels)

In [69]:
class_weights = [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(class_weights, dtype=torch.float)

In [70]:
sample_weights = [class_weights[label].item() for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [71]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [72]:
class AttentionBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.attn(x)

In [73]:
class HybridModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Multi-scale conv
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(1, 16, 5, padding=2)

        self.pool = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.attn = AttentionBlock(64)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x1 = torch.relu(self.conv1(x))
        x2 = torch.relu(self.conv3(x))

        x = torch.cat([x1, x2], dim=1)
        x = self.pool(x)

        x = torch.relu(self.conv2(x))
        x = self.attn(x)
        x = self.pool(x)

        x = self.fc(x)
        return x

model = HybridModel(num_classes)

In [74]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [75]:
MODEL_PATH = "hybrid_model.pth"

In [76]:
if can_train:
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'num_classes': num_classes
    }, MODEL_PATH)

In [77]:
checkpoint = torch.load(MODEL_PATH, map_location=torch.device("cpu"))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

HybridModel(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (attn): AttentionBlock(
    (attn): Sequential(
      (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): Sigmoid()
    )
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=16384, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)

In [78]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy())

In [79]:
print("\n===== FINAL METRICS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='weighted'))
print("Recall   :", recall_score(y_true, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_true, y_pred, average='weighted'))


===== FINAL METRICS =====
Accuracy : 0.8431667437914546
Precision: 0.9117963741862458
Recall   : 0.8431667437914546
F1 Score : 0.8577250299583973


In [80]:
print("\n===== SAMPLE PREDICTIONS =====")

for i in random.sample(range(len(test_dataset)), 3):
    img, label = test_dataset[i]
    with torch.no_grad():
        pred = model(img.unsqueeze(0))
        _, pred_class = torch.max(pred, 1)

    print("\nActual   :", class_names[label])
    print("Predicted:", class_names[pred_class.item()])


===== SAMPLE PREDICTIONS =====

Actual   : Non Demented
Predicted: Non Demented

Actual   : Non Demented
Predicted: Non Demented

Actual   : Very mild Dementia
Predicted: Very mild Dementia
